# Rounded Bottom / Rounded Top on SPY
## Strategy Brief
The Rounded Bottom / Rounded Top strategy identifies potential reversal points in the SPY ETF by detecting rounded price formations. A rounded bottom suggests a bullish reversal, while a rounded top indicates a bearish reversal. The strategy enters a long position after a rounded bottom and a short position after a rounded top. Historical analysis shows mixed results, with periods of strong performance during clear market trends.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the trading parameters and constants used throughout the strategy. This includes the lookback period for detecting rounded formations and other relevant thresholds.

In [ ]:
LOOKBACK_PERIOD = 20
THRESHOLD = 0.02
START_DATE = '2010-01-01'
END_DATE = 'today'
TICKER = 'SPY'

## PHASE 2 - Data Exploration
We will download historical SPY data using yfinance, compute the necessary indicators to identify rounded formations, and visualize these indicators overlaid on the price chart.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Compute moving averages
rolling_mean = data['Close'].rolling(window=LOOKBACK_PERIOD).mean()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(rolling_mean, label=f'{LOOKBACK_PERIOD}-day Rolling Mean', linestyle='--')
plt.title('SPY Price with Rounded Bottom/Top Indicator')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
This phase involves creating a signal series based on the detection of rounded bottoms and tops, defining entry and exit logic, and generating a positions series.

In [ ]:
def detect_rounded_formations(prices, threshold):
    diff = prices.diff()
    rounded_bottom = (diff < -threshold).rolling(window=LOOKBACK_PERIOD).sum() == LOOKBACK_PERIOD
    rounded_top = (diff > threshold).rolling(window=LOOKBACK_PERIOD).sum() == LOOKBACK_PERIOD
    return rounded_bottom, rounded_top

rounded_bottom, rounded_top = detect_rounded_formations(rolling_mean, THRESHOLD)

# Create signals
signals = pd.Series(index=data.index, data=0)
signals[rounded_bottom] = 1
signals[rounded_top] = -1

# Generate positions
positions = signals.shift(1).fillna(0)

## PHASE 4 - Coding & Backtesting
We will backtest the strategy by computing daily returns based on the positions, and plot the resulting equity curve.

In [ ]:
daily_returns = data['Close'].pct_change()
strategy_returns = positions * daily_returns

# Compute equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.title('Equity Curve of Rounded Bottom/Top Strategy')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
Evaluate the strategy's performance using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. Compare these metrics against a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(returns):
    cagr = (equity_curve.iloc[-1] ** (252 / len(returns))) - 1
    sharpe_ratio = returns.mean() / returns.std() * np.sqrt(252)
    downside_std = returns[returns < 0].std()
    sortino_ratio = returns.mean() / downside_std * np.sqrt(252)
    max_drawdown = (equity_curve.cummax() - equity_curve).max()
    calmar_ratio = cagr / max_drawdown
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_metrics = calculate_performance_metrics(strategy_returns)
buy_and_hold_metrics = calculate_performance_metrics(daily_returns)

# Display comparison
df_comparison = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy & Hold': buy_and_hold_metrics
})
print(df_comparison)

## PHASE 6 - Deploy & Monitor
Create a function to download the last 60 days of SPY data, compute today's signal, and print the recommended position.

In [ ]:
def get_latest_signal(ticker, lookback_period, threshold):
    recent_data = yf.download(ticker, period='60d')
    rolling_mean_recent = recent_data['Close'].rolling(window=lookback_period).mean()
    rounded_bottom, rounded_top = detect_rounded_formations(rolling_mean_recent, threshold)
    latest_signal = 0
    if rounded_bottom.iloc[-1]:
        latest_signal = 1
    elif rounded_top.iloc[-1]:
        latest_signal = -1
    return latest_signal

latest_signal = get_latest_signal(TICKER, LOOKBACK_PERIOD, THRESHOLD)
print(f'Today\'s recommended position for {TICKER}: {"Long" if latest_signal == 1 else "Short" if latest_signal == -1 else "Neutral"}')